# Lesson 10.2 · Sanity checks

The worst mistakes in a project do not raise exceptions: they produce **plausible numbers**. This lesson collects three checks that run in a few minutes and that expose most of the defects you met in the type E exercises.

**What you will learn**

- The **permuted labels test**: a correct flow on random labels must give AUC 0.5.
- The **learning curve**: understanding whether you need more data, more capacity or more features.
- Validation that **respects the structure** of the data (groups, time, areas).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve
from sklearn.pipeline import make_pipeline

df = ds.loans(n=1500, seed=4)
y = df.pop("defaulted").to_numpy()
numeric = ["age", "income", "job_tenure", "amount"]
X = pd.get_dummies(df[numeric + ["city", "contract_type"]], columns=["city", "contract_type"], dtype=float)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
forest = lambda: RandomForestClassifier(200, n_jobs=-1, random_state=0)

## Check 1 · permuted labels

If you shuffle the labels at random **before** the whole flow, there is no signal left to find: a correct flow must give an AUC close to 0.5. If you get much more, information is coming in through the wrong door (leakage, duplicates between training and validation, feature selection on the whole dataset...).

Here the **correct** flow is a pipeline evaluated in cross-validation; the **faulty** flow duplicates the rare class *before* the cross-validation, so the same rows end up both in the training and in the validation set.

In [ ]:
rng = np.random.default_rng(0)
y_perm = rng.permutation(y)                                    # no signal: X and y_perm are independent

correct_pipeline = make_pipeline(SimpleImputer(strategy="median"), forest())
correct = cross_val_score(correct_pipeline, X, y_perm, cv=cv, scoring="roc_auc").mean()


def cv_with_duplicates_first(X, y):
    rare = np.where(y == 1)[0]
    extra = np.random.default_rng(0).choice(rare, (y == 0).sum() - len(rare), replace=True)       # 'balance' by duplicating the rare class
    idx = np.r_[np.arange(len(y)), extra]
    Xd = SimpleImputer(strategy="median").fit_transform(X)[idx]
    return cross_val_score(forest(), Xd, y[idx], cv=cv, scoring="roc_auc").mean()


faulty = cv_with_duplicates_first(X, y_perm)
print(f"AUC on permuted labels | correct flow: {correct:.3f} | duplicates before the CV: {faulty:.3f}")

The correct flow gives about 0.5, as it should. The faulty one gives **over 0.9 on labels with no meaning at all**: the forest memorized the duplicated rows and recognizes them in the validation fold. It is the cheapest test there is against leakage: **if your flow beats chance on data with no signal, it
is not a valid flow**.

## Check 2 · learning curve

Train on the same problem with growing amounts of training data and look at what the training score and the validation score do.

In [ ]:
sizes, tr_scores, va_scores = learning_curve(make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(200, min_samples_leaf=3, n_jobs=-1, random_state=0)),
                                             X, y, cv=cv, scoring="roc_auc", train_sizes=[0.1, 0.25, 0.5, 0.75, 1.0], random_state=0)
curve = pd.DataFrame({"training": tr_scores.mean(1), "validation": va_scores.mean(1)}, index=sizes.astype(int))
curve.index.name = "training examples"
curve.round(3)

How to read it:

- **Validation still climbing** as the data grow: collecting more examples helps.
- **Validation flat and training much higher**: overfitting; you need regularization or a simpler model.
- **The two curves close together and low**: underfitting; you need better features or a more capable model.

Here the validation climbs slowly (from 0.74 with 120 examples to 0.79 with 1200) while the training stays at 0.98: the gap of 0.2 is typical of a deep forest that memorizes the training set, and doubling the examples would give a few thousandths. You need better features or regularization (`min_samples_leaf`), not fine tuning.

## Check 3 · the validation must look like the real use

If in production you predict **new patients**, **future months** or **areas never seen**, the validation must simulate exactly that: `GroupKFold` for groups, a temporal split for series, geographic blocks for maps. With a random partition the "neighboring" examples (same patient, same month, same neighborhood) are both in the training and in the validation set and the error
comes out much lower than the one you will really see. In project P2 you measure this effect on California housing: the gap is 40%.

> **Common mistake.** Attributing a good validation score to the quality of the model, without asking whether the validation protocol looks like how the model will be used.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>What does an AUC of 0.9 on the permuted labels test tell you?</summary>

That information about the label is entering the flow through a path that should not exist: duplicates between training and validation, preprocessing or feature selection fitted on the whole dataset, features built with the target. The model has not 'learned' anything: it has seen the answers again.

</details>

<details><summary>In what order do you run the checks?</summary>

First the cheap and general ones (baseline, permuted labels), then those on the protocol (validation that respects the structure of the data), then the learning curves and the error analyses. A check that fails has to be fixed before tuning the model: optimizing a faulty flow is wasted time.

</details>